# Introduction
Included in this script is:

1.) The function we use to determine the dynamical habitability of our injected planets and export those results 

2.) The function calls for each system and each of its given architectures

In [1]:
# Import Statements
import rebound
import math
import numpy as np
import matplotlib.pyplot as plt
import time
import astropy
import pandas as pd
import os
import scipy
%matplotlib inline

In [2]:
# Constants for all files/needed for export

## Max time of each sim on file
tmax = 1e6
## Num Test Particles in each sim on file
num_particles = 100

## Constants for both eq's
LSUN = 3.828*10**26 # watts
AU_METERS = 1.495979*10**11 # meters
F_EARTH = 1366 # watts*m^-2
F_INNER = F_EARTH * 1.7
F_OUTER = F_EARTH * 0.3

## Date for exporting
date = time.strftime('%Y-%m-%d', time.localtime())

In [3]:
def find_hab_results(host_star, deg, directory, known_bodies, r_star, L_LSUN, m_star):
    # Calculate the radius of the star in AU
    r_star_au = r_star * 0.0045624726
    # Calculate the luminosity of the star in watts
    L_STAR = L_LSUN*LSUN
    
    # Calculate HZ Lims
    INNER_HZ = np.sqrt(L_STAR/(4*math.pi*(F_INNER)))/AU_METERS
    OUTER_HZ = np.sqrt(L_STAR/(4*math.pi*(F_OUTER)))/AU_METERS
    
    # Calculate ejection limit in AU
    sma_ejection_limit = OUTER_HZ*2 

    # Calculate Roche Limit
    rho = m_star/(4/3*math.pi*r_star**3) * 5.91
    roche_limit = 2.44*r_star_au * ((2.81/5.5))**(1/3)
    
    # Prep final dataframes that will store habitability and ejection results
    semi_major_axes = np.linspace(INNER_HZ, OUTER_HZ, num_particles)
    all_habitability_results = pd.DataFrame(columns = semi_major_axes)
    all_ejection_results = pd.DataFrame(columns = semi_major_axes)
    
    # Find Survival and Ejection
    for path, folders, files in os.walk(directory):
        for filename in files:    
            # Declare dataframe with recorded sim info from file
            stored_vals = pd.read_csv(directory + filename, sep = ',')
            
            # Declare dataframe to record survival
            survival = pd.DataFrame({'body':[], 'starting_a':[], 'starting_e':[], 'primary':[], 'ejected': [], 'remains_habitable': [], 't_uninhab':[]})
 
            # Add in other ejection limits, eccentricity gets too high, velocity too high, ect.
            for particle_index in range(len(stored_vals.body[stored_vals.time ==tmax])):
                # Skip all known bodies in the system
                if particle_index <= known_bodies - 1:
                    continue
                # Get a temporary dataframe for the current body being tracked
                temp_df = stored_vals[stored_vals.body == particle_index]
            
                # Tracks if the planet ever gets ejected
                borked = False
                uninhabitable = False
                time_temp = 0
                
                # Check if the planet has either been ejected (a is negative) or engulfed by the star (periastron is less than the stellar radius)
                if not (temp_df.a.iloc[-1] <= 0 or (temp_df.a.iloc[-1]*(1-temp_df.e.iloc[-1])) < r_star_au):
                    # If not, loop through each value in the dataframe for the particle to find if it ever went uninhabitable
                    for index, row in temp_df.iterrows():
                        ## Equation for flux chosen from Bolmont
                        e_corrected_flux = (L_STAR/(4*np.pi*(row.a*AU_METERS)**2*np.sqrt(1-row.e**2)))/F_EARTH
            
                        # Make sure the flux found is a real object
                        if not np.isrealobj(e_corrected_flux):
                            continue
                        # Checks if the planet ever goes uninhabitable, if it does, track that it goes uninhabitable
                        if (e_corrected_flux > 1.7 or e_corrected_flux < 0.3) and uninhabitable == False and borked == False:
                            time_temp = row.time
                            uninhabitable = True
                        # Check if the planet is ever dynamically ustable, if yes, track that and break out of the loop
                        if (row.a > sma_ejection_limit or row.a*(1-row.e) < roche_limit or row.e > 1) and borked == False: 
                            time_temp = row.time
                            borked = True
                            break
                    # Add determinations to the dataframe
                    #  time_temp tracks the time at which the plane would become dynamically unstable or uninhabitable;
                    #  if the planet remains inhabitable, the 
                    if borked:
                        survival.loc[len(survival)] = [particle_index, temp_df.starting_a.iloc[0], temp_df.e.iloc[0], 0, True, False, time_temp]
                    elif uninhabitable:
                        survival.loc[len(survival)] = [particle_index, temp_df.starting_a.iloc[0], temp_df.e.iloc[0], 0, False, False, time_temp]
                    else:
                        survival.loc[len(survival)] = [particle_index, temp_df.starting_a.iloc[0], temp_df.e.iloc[0], 0, False, True, -1]
                    
                else:
                    survival.loc[len(survival)] = [particle_index, temp_df.starting_a.iloc[0], temp_df.e.iloc[0], 0, True, False, 0]
            # For testing purposes: check habitability, dynamic instability, and uninhabitability statistics
            print('Sim:', filename, '\tHabitable:', len(survival[survival.remains_habitable == True]), '\tEjected:', len(survival[survival.ejected == True]))

            # Add survival and habitability results to separate dataframes
            all_habitability_results.loc[len(all_habitability_results)] = survival.remains_habitable.to_numpy()
            all_ejection_results.loc[len(all_ejection_results)] = survival.ejected.to_numpy()

    # Set the filename
    export_name_metadata = date +'_host_star=' + host_star + deg + 'deg' + '_num_bodies=' + str(known_bodies) + '_tmax=' + str(tmax) + '_num_sims=' + str(all_habitability_results.shape[0])

    # Export dataframes as csv's
    all_habitability_results.to_csv('results/habitability_'+ export_name_metadata + '.csv', index=False)
    all_ejection_results.to_csv('results/ejection_'+ export_name_metadata + '.csv', index=False)

In [4]:
find_hab_results('36OphA_june', '0' ,'results/36OphA_0deg_june/', 2, 0.817, 0.326, 0.867)

Sim: test_id_num-0_36OphAIAS15_2026_06_06_01__inclination-0.0deg.csv 	Habitable: 98 	Ejected: 0
Sim: test_id_num-10_36OphAIAS15_2026_06_06_01__inclination-0.0deg.csv 	Habitable: 98 	Ejected: 0
Sim: test_id_num-11_36OphAIAS15_2026_06_06_16__inclination-0.0deg.csv 	Habitable: 98 	Ejected: 0
Sim: test_id_num-12_36OphAIAS15_2026_06_07_07__inclination-0.0deg.csv 	Habitable: 99 	Ejected: 0
Sim: test_id_num-13_36OphAIAS15_2026_06_07_22__inclination-0.0deg.csv 	Habitable: 98 	Ejected: 0
Sim: test_id_num-14_36OphAIAS15_2026_06_08_13__inclination-0.0deg.csv 	Habitable: 98 	Ejected: 0
Sim: test_id_num-15_36OphAIAS15_2026_06_06_01__inclination-0.0deg.csv 	Habitable: 98 	Ejected: 0
Sim: test_id_num-16_36OphAIAS15_2026_06_06_16__inclination-0.0deg.csv 	Habitable: 98 	Ejected: 0
Sim: test_id_num-17_36OphAIAS15_2026_06_07_06__inclination-0.0deg.csv 	Habitable: 98 	Ejected: 0
Sim: test_id_num-18_36OphAIAS15_2026_06_07_21__inclination-0.0deg.csv 	Habitable: 99 	Ejected: 0
Sim: test_id_num-19_36OphAIAS15

In [5]:
find_hab_results('36OphA_july', '45', 'results/36OphA_45deg_july/', 2, 0.817, 0.326, 0.867)

Sim: test_id_num-0_36OphAIAS15_2026_06_29_22__inclination-45.0deg.csv 	Habitable: 95 	Ejected: 0
Sim: test_id_num-10_36OphAIAS15_2026_07_02_23__inclination-45.0deg.csv 	Habitable: 92 	Ejected: 2
Sim: test_id_num-11_36OphAIAS15_2026_07_04_11__inclination-45.0deg.csv 	Habitable: 94 	Ejected: 1
Sim: test_id_num-12_36OphAIAS15_2026_06_29_23__inclination-45.0deg.csv 	Habitable: 96 	Ejected: 0
Sim: test_id_num-13_36OphAIAS15_2026_07_01_12__inclination-45.0deg.csv 	Habitable: 94 	Ejected: 0
Sim: test_id_num-14_36OphAIAS15_2026_07_02_22__inclination-45.0deg.csv 	Habitable: 95 	Ejected: 1
Sim: test_id_num-15_36OphAIAS15_2026_07_04_07__inclination-45.0deg.csv 	Habitable: 96 	Ejected: 0
Sim: test_id_num-16_36OphAIAS15_2026_06_29_23__inclination-45.0deg.csv 	Habitable: 94 	Ejected: 0
Sim: test_id_num-17_36OphAIAS15_2026_07_01_10__inclination-45.0deg.csv 	Habitable: 93 	Ejected: 1
Sim: test_id_num-18_36OphAIAS15_2026_07_02_22__inclination-45.0deg.csv 	Habitable: 96 	Ejected: 0
Sim: test_id_num-19_3

In [6]:
find_hab_results('36OphB_july', '0', 'results/36OphB_0deg_july/', 2, 0.718, 0.328, 0.803)

Sim: test_id_num-0_36OphAIAS15_2026_06_09_10__inclination-0.0deg.csv 	Habitable: 98 	Ejected: 0
Sim: test_id_num-10_36OphAIAS15_2026_06_10_19__inclination-0.0deg.csv 	Habitable: 99 	Ejected: 0
Sim: test_id_num-11_36OphAIAS15_2026_06_11_12__inclination-0.0deg.csv 	Habitable: 98 	Ejected: 0
Sim: test_id_num-12_36OphAIAS15_2026_06_09_10__inclination-0.0deg.csv 	Habitable: 98 	Ejected: 0
Sim: test_id_num-13_36OphAIAS15_2026_06_10_03__inclination-0.0deg.csv 	Habitable: 98 	Ejected: 0
Sim: test_id_num-14_36OphAIAS15_2026_06_10_20__inclination-0.0deg.csv 	Habitable: 98 	Ejected: 0
Sim: test_id_num-15_36OphAIAS15_2026_06_11_13__inclination-0.0deg.csv 	Habitable: 98 	Ejected: 0
Sim: test_id_num-16_36OphAIAS15_2026_06_09_10__inclination-0.0deg.csv 	Habitable: 97 	Ejected: 0
Sim: test_id_num-17_36OphAIAS15_2026_06_10_03__inclination-0.0deg.csv 	Habitable: 97 	Ejected: 0
Sim: test_id_num-18_36OphAIAS15_2026_06_10_20__inclination-0.0deg.csv 	Habitable: 98 	Ejected: 0
Sim: test_id_num-19_36OphAIAS15

In [8]:
find_hab_results('36OphB', '45', 'results/36OphB_45deg_july/', 2, 0.718, 0.328, 0.7803)

Sim: test_id_num-0_36OphAIAS15_2026_06_13_03__inclination-45.0deg.csv 	Habitable: 94 	Ejected: 0
Sim: test_id_num-10_36OphAIAS15_2026_06_16_04__inclination-45.0deg.csv 	Habitable: 95 	Ejected: 0
Sim: test_id_num-11_36OphAIAS15_2026_06_27_22__inclination-45.0deg.csv 	Habitable: 95 	Ejected: 0
Sim: test_id_num-12_36OphAIAS15_2026_06_13_03__inclination-45.0deg.csv 	Habitable: 95 	Ejected: 0
Sim: test_id_num-13_36OphAIAS15_2026_06_14_15__inclination-45.0deg.csv 	Habitable: 94 	Ejected: 1
Sim: test_id_num-14_36OphAIAS15_2026_06_16_01__inclination-45.0deg.csv 	Habitable: 94 	Ejected: 0
Sim: test_id_num-15_36OphAIAS15_2026_06_28_00__inclination-45.0deg.csv 	Habitable: 96 	Ejected: 0
Sim: test_id_num-16_36OphAIAS15_2026_06_13_07__inclination-45.0deg.csv 	Habitable: 93 	Ejected: 1
Sim: test_id_num-17_36OphAIAS15_2026_06_14_18__inclination-45.0deg.csv 	Habitable: 93 	Ejected: 0
Sim: test_id_num-18_36OphAIAS15_2026_06_16_07__inclination-45.0deg.csv 	Habitable: 95 	Ejected: 0
Sim: test_id_num-19_3

In [4]:
find_hab_results('GamLeoA', '45', 'results/GamLeoA_noC_45deg/', 3, 26.08, 251,1.66)

Sim: test_id_num-0_GamLeoAias15_2026_07_20_19__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: test_id_num-10_GamLeoAias15_2026_07_21_05__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: test_id_num-11_GamLeoAias15_2026_07_21_11__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: test_id_num-12_GamLeoAias15_2026_07_20_20__inclination-45deg.csv 	Habitable: 0 	Ejected: 100


/tmp/ipykernel_2485/2174087151.py:50: RuntimeWarning: invalid value encountered in sqrt
  e_corrected_flux = (L_STAR/(4*np.pi*(row.a*AU_METERS)**2*np.sqrt(1-row.e**2)))/F_EARTH


Sim: test_id_num-13_GamLeoAias15_2026_07_21_03__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: test_id_num-14_GamLeoAias15_2026_07_21_08__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: test_id_num-15_GamLeoAias15_2026_07_21_11__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: test_id_num-16_GamLeoAias15_2026_07_21_00__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: test_id_num-17_GamLeoAias15_2026_07_21_04__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: test_id_num-18_GamLeoAias15_2026_07_21_08__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: test_id_num-19_GamLeoAias15_2026_07_21_12__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: test_id_num-1_GamLeoAias15_2026_07_21_00__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: test_id_num-20_GamLeoAias15_2026_07_20_19__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: test_id_num-21_GamLeoAias15_2026_07_21_00__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: test_id_num-22_G

/tmp/ipykernel_2485/2174087151.py:50: RuntimeWarning: invalid value encountered in sqrt
  e_corrected_flux = (L_STAR/(4*np.pi*(row.a*AU_METERS)**2*np.sqrt(1-row.e**2)))/F_EARTH


Sim: test_id_num-45_GamLeoAias15_2026_07_21_06__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: test_id_num-46_GamLeoAias15_2026_07_21_10__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: test_id_num-47_GamLeoAias15_2026_07_21_15__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: test_id_num-48_GamLeoAias15_2026_07_20_20__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: test_id_num-49_GamLeoAias15_2026_07_21_02__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: test_id_num-4_GamLeoAias15_2026_07_20_19__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: test_id_num-50_GamLeoAias15_2026_07_21_07__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: test_id_num-51_GamLeoAias15_2026_07_21_10__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: test_id_num-52_GamLeoAias15_2026_07_20_20__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: test_id_num-53_GamLeoAias15_2026_07_21_00__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: test_id_num-54_G

/tmp/ipykernel_2485/2174087151.py:50: RuntimeWarning: invalid value encountered in sqrt
  e_corrected_flux = (L_STAR/(4*np.pi*(row.a*AU_METERS)**2*np.sqrt(1-row.e**2)))/F_EARTH


Sim: test_id_num-60_GamLeoAias15_2026_07_20_21__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: test_id_num-61_GamLeoAias15_2026_07_21_01__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: test_id_num-62_GamLeoAias15_2026_07_21_05__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: test_id_num-63_GamLeoAias15_2026_07_21_09__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: test_id_num-64_GamLeoAias15_2026_07_20_20__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: test_id_num-65_GamLeoAias15_2026_07_21_00__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: test_id_num-66_GamLeoAias15_2026_07_21_04__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: test_id_num-67_GamLeoAias15_2026_07_21_08__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: test_id_num-68_GamLeoAias15_2026_07_20_19__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: test_id_num-69_GamLeoAias15_2026_07_21_00__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: test_id_num-6_G

/tmp/ipykernel_2485/2174087151.py:50: RuntimeWarning: invalid value encountered in sqrt
  e_corrected_flux = (L_STAR/(4*np.pi*(row.a*AU_METERS)**2*np.sqrt(1-row.e**2)))/F_EARTH


Sim: test_id_num-86_GamLeoAias15_2026_07_21_05__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: test_id_num-87_GamLeoAias15_2026_07_21_09__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: test_id_num-88_GamLeoAias15_2026_07_20_16__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: test_id_num-89_GamLeoAias15_2026_07_20_20__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: test_id_num-8_GamLeoAias15_2026_07_20_20__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: test_id_num-90_GamLeoAias15_2026_07_21_00__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: test_id_num-91_GamLeoAias15_2026_07_21_05__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: test_id_num-92_GamLeoAias15_2026_07_20_16__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: test_id_num-93_GamLeoAias15_2026_07_20_20__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: test_id_num-94_GamLeoAias15_2026_07_21_00__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: test_id_num-95_G

In [5]:
find_hab_results('GamLeoA_withC', '45', 'results/GamLeoA_withC_45deg/', 4, 26.08, 251,1.66)

Sim: withCtest_id_num-0_GamLeoAias15_2026_07_22_15__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: withCtest_id_num-10_GamLeoAias15_2026_07_23_00__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: withCtest_id_num-11_GamLeoAias15_2026_07_23_03__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: withCtest_id_num-12_GamLeoAias15_2026_07_22_15__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: withCtest_id_num-13_GamLeoAias15_2026_07_22_19__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: withCtest_id_num-14_GamLeoAias15_2026_07_23_00__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: withCtest_id_num-15_GamLeoAias15_2026_07_23_06__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: withCtest_id_num-16_GamLeoAias15_2026_07_22_15__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: withCtest_id_num-17_GamLeoAias15_2026_07_22_19__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: withCtest_id_num-18_GamLeoAias15_2026_07_23_01__inclination-45deg.csv

In [8]:
find_hab_results('GamLeoB', '45', 'results/GamLeoB_noC_45deg/', 3, 10.55, 63, 1.55)

Sim: test_id_num-0_GamLeoBias15_2026_07_21_21__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: test_id_num-10_GamLeoBias15_2026_07_22_06__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: test_id_num-11_GamLeoBias15_2026_07_22_10__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: test_id_num-12_GamLeoBias15_2026_07_21_21__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: test_id_num-13_GamLeoBias15_2026_07_22_02__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: test_id_num-14_GamLeoBias15_2026_07_22_07__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: test_id_num-15_GamLeoBias15_2026_07_22_11__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: test_id_num-16_GamLeoBias15_2026_07_21_21__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: test_id_num-17_GamLeoBias15_2026_07_22_01__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: test_id_num-18_GamLeoBias15_2026_07_22_06__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: test_id_num-19_G

In [9]:
find_hab_results('GamLeoB_withC', '45', 'results/GamLeoB_withC_45deg/', 4, 10.55, 63, 1.55)

Sim: withCtest_id_num-0_GamLeoBias15_2026_07_25_21__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: withCtest_id_num-10_GamLeoBias15_2026_07_25_21__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: withCtest_id_num-11_GamLeoBias15_2026_07_26_01__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: withCtest_id_num-12_GamLeoBias15_2026_07_26_05__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: withCtest_id_num-13_GamLeoBias15_2026_07_26_09__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: withCtest_id_num-14_GamLeoBias15_2026_07_26_13__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: withCtest_id_num-15_GamLeoBias15_2026_07_26_17__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: withCtest_id_num-16_GamLeoBias15_2026_07_26_21__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: withCtest_id_num-17_GamLeoBias15_2026_07_27_03__inclination-45deg.csv 	Habitable: 0 	Ejected: 100
Sim: withCtest_id_num-18_GamLeoBias15_2026_07_27_07__inclination-45deg.csv

In [4]:
find_hab_results('70OphA', '0', 'results/70OphA_0deg_aug/', 2, 0.8310, 0.53, 0.8827)

Sim: test_id_num-0_70OphA70OphA_2026_07_05_00__inclination-0.0deg.csv 	Habitable: 99 	Ejected: 0
Sim: test_id_num-10_70OphA70OphA_2026_07_05_22__inclination-0.0deg.csv 	Habitable: 98 	Ejected: 0
Sim: test_id_num-11_70OphA70OphA_2026_07_06_10__inclination-0.0deg.csv 	Habitable: 98 	Ejected: 0
Sim: test_id_num-12_70OphA70OphA_2026_07_05_01__inclination-0.0deg.csv 	Habitable: 98 	Ejected: 0
Sim: test_id_num-13_70OphA70OphA_2026_07_05_12__inclination-0.0deg.csv 	Habitable: 98 	Ejected: 0
Sim: test_id_num-14_70OphA70OphA_2026_07_05_23__inclination-0.0deg.csv 	Habitable: 99 	Ejected: 0
Sim: test_id_num-15_70OphA70OphA_2026_07_06_10__inclination-0.0deg.csv 	Habitable: 98 	Ejected: 0
Sim: test_id_num-16_70OphA70OphA_2026_07_05_00__inclination-0.0deg.csv 	Habitable: 98 	Ejected: 0
Sim: test_id_num-17_70OphA70OphA_2026_07_05_11__inclination-0.0deg.csv 	Habitable: 98 	Ejected: 0
Sim: test_id_num-18_70OphA70OphA_2026_07_05_22__inclination-0.0deg.csv 	Habitable: 98 	Ejected: 0
Sim: test_id_num-19_7

In [5]:
find_hab_results('70OphA', '45', 'results/70OphA_45deg_aug/', 2, 0.8310, 0.53, 0.8827)

Sim: test_id_num-0_70OphA70OphA_2026_07_29_09__inclination-45.0deg.csv 	Habitable: 94 	Ejected: 0
Sim: test_id_num-10_70OphA70OphA_2026_07_30_23__inclination-45.0deg.csv 	Habitable: 95 	Ejected: 0
Sim: test_id_num-11_70OphA70OphA_2026_07_31_18__inclination-45.0deg.csv 	Habitable: 95 	Ejected: 0
Sim: test_id_num-12_70OphA70OphA_2026_07_29_09__inclination-45.0deg.csv 	Habitable: 95 	Ejected: 0
Sim: test_id_num-13_70OphA70OphA_2026_07_30_04__inclination-45.0deg.csv 	Habitable: 95 	Ejected: 0
Sim: test_id_num-14_70OphA70OphA_2026_07_30_23__inclination-45.0deg.csv 	Habitable: 95 	Ejected: 0
Sim: test_id_num-15_70OphA70OphA_2026_07_31_18__inclination-45.0deg.csv 	Habitable: 95 	Ejected: 0
Sim: test_id_num-16_70OphA70OphA_2026_07_29_09__inclination-45.0deg.csv 	Habitable: 94 	Ejected: 0
Sim: test_id_num-17_70OphA70OphA_2026_07_30_03__inclination-45.0deg.csv 	Habitable: 95 	Ejected: 0
Sim: test_id_num-18_70OphA70OphA_2026_07_30_22__inclination-45.0deg.csv 	Habitable: 95 	Ejected: 0
Sim: test_i

In [6]:
find_hab_results('70OphB', '0', 'results/70OphB_0deg_aug/', 2, 0.6697, 0.15, 0.7319)

Sim: test_id_num-0_70OphB70OphB_2026_07_09_11__inclination-0.0deg.csv 	Habitable: 98 	Ejected: 0
Sim: test_id_num-10_70OphB70OphB_2026_07_11_15__inclination-0.0deg.csv 	Habitable: 99 	Ejected: 0
Sim: test_id_num-11_70OphB70OphB_2026_07_12_17__inclination-0.0deg.csv 	Habitable: 98 	Ejected: 0
Sim: test_id_num-12_70OphB70OphB_2026_07_09_13__inclination-0.0deg.csv 	Habitable: 98 	Ejected: 0
Sim: test_id_num-13_70OphB70OphB_2026_07_10_14__inclination-0.0deg.csv 	Habitable: 99 	Ejected: 0
Sim: test_id_num-14_70OphB70OphB_2026_07_11_16__inclination-0.0deg.csv 	Habitable: 98 	Ejected: 0
Sim: test_id_num-15_70OphB70OphB_2026_07_12_16__inclination-0.0deg.csv 	Habitable: 98 	Ejected: 0
Sim: test_id_num-16_70OphB70OphB_2026_07_09_12__inclination-0.0deg.csv 	Habitable: 98 	Ejected: 0
Sim: test_id_num-17_70OphB70OphB_2026_07_10_15__inclination-0.0deg.csv 	Habitable: 98 	Ejected: 0
Sim: test_id_num-18_70OphB70OphB_2026_07_11_18__inclination-0.0deg.csv 	Habitable: 99 	Ejected: 0
Sim: test_id_num-19_7

In [7]:
find_hab_results('70OphB', '45', 'results/70OphB_45deg/', 2, 0.6697, 0.15, 0.7319)

Sim: test_id_num-0_70OphB70OphB_2026_03_18_02__inclination-45.0deg.csv 	Habitable: 95 	Ejected: 0
Sim: test_id_num-10_70OphB70OphB_2026_03_25_10__inclination-45.0deg.csv 	Habitable: 95 	Ejected: 0
Sim: test_id_num-11_70OphB70OphB_2026_03_18_02__inclination-45.0deg.csv 	Habitable: 96 	Ejected: 0
Sim: test_id_num-12_70OphB70OphB_2026_03_19_23__inclination-45.0deg.csv 	Habitable: 96 	Ejected: 0
Sim: test_id_num-13_70OphB70OphB_2026_03_21_19__inclination-45.0deg.csv 	Habitable: 96 	Ejected: 0
Sim: test_id_num-14_70OphB70OphB_2026_03_23_16__inclination-45.0deg.csv 	Habitable: 95 	Ejected: 0
Sim: test_id_num-15_70OphB70OphB_2026_03_25_10__inclination-45.0deg.csv 	Habitable: 95 	Ejected: 0
Sim: test_id_num-16_70OphB70OphB_2026_03_18_02__inclination-45.0deg.csv 	Habitable: 94 	Ejected: 0
Sim: test_id_num-17_70OphB70OphB_2026_03_19_23__inclination-45.0deg.csv 	Habitable: 96 	Ejected: 0
Sim: test_id_num-18_70OphB70OphB_2026_03_21_18__inclination-45.0deg.csv 	Habitable: 95 	Ejected: 0
Sim: test_i